In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [ ]:
data = np.load("pems04.npz")

traffic = data["data"][:,:,2]

print(traffic.shape)

(16992, 307)


In [ ]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

In [ ]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    - sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(16980, 12, 307)
(16980, 307)


In [ ]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [ ]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [ ]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [ ]:
num_nodes = 307
num_hyperedges = 16

node_embeddings = np.random.randn(
    num_nodes,
    16
)

kmeans = KMeans(
    n_clusters=num_hyperedges,
    random_state=42
)

clusters = kmeans.fit_predict(
    node_embeddings
)

H = np.zeros(
    (
        num_nodes,
        num_hyperedges
    )
)

for node, cluster in enumerate(clusters):

    H[node, cluster] = 1

H = torch.tensor(
    H,
    dtype=torch.float32
)

print(H.shape)

c:\Users\student\paperTest\venv\Lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


torch.Size([307, 16])


In [ ]:
class MultiScaleTemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv3 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

        self.conv5 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(5,1),
            padding=(2,0)
        )

        self.conv7 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(7,1),
            padding=(3,0)
        )

    def forward(self,x):

        out3 = self.conv3(x)

        out5 = self.conv5(x)

        out7 = self.conv7(x)

        return torch.relu(
            out3 + out5 + out7
        )

In [ ]:
class AdaptiveGraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        channels
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        adaptive_adj = torch.softmax(
            torch.relu(
                self.node_embeddings
                @
                self.node_embeddings.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            adaptive_adj,
            x
        )

        x = x.permute(
            0,2,3,1
        )

        x = self.weight(x)

        x = x.permute(
            0,3,1,2
        )

        return torch.relu(x)

In [ ]:
class ClusterHypergraphConv(nn.Module):

    def __init__(
        self,
        channels,
        H
    ):

        super().__init__()

        self.H = H

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        H = self.H.to(
            x.device
        )

        hyper_adj = H @ H.T

        hyper_adj = hyper_adj / (
            hyper_adj.sum(
                dim=1,
                keepdim=True
            )
            + 1e-6
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            hyper_adj,
            x
        )

        x = x.permute(
            0,2,3,1
        )

        x = self.weight(x)

        x = x.permute(
            0,3,1,2
        )

        return torch.relu(x)

In [ ]:
class CAHSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = MultiScaleTemporalConv(
            1,
            32
        )

        self.graph = AdaptiveGraphConv(
            num_nodes=307,
            channels=32
        )

        self.hypergraph = ClusterHypergraphConv(
            channels=32,
            H=H
        )

        self.temp2 = MultiScaleTemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        graph_features = self.graph(x)

        hyper_features = self.hypergraph(x)

        x = graph_features + hyper_features

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        return x.squeeze(-1)

In [ ]:
model = CAHSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 307])
torch.Size([64, 307])


In [ ]:
model = CAHSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.023763
Epoch 2/30 Loss: 0.003410
Epoch 3/30 Loss: 0.001728
Epoch 4/30 Loss: 0.001224
Epoch 5/30 Loss: 0.001112
Epoch 6/30 Loss: 0.001055
Epoch 7/30 Loss: 0.001037
Epoch 8/30 Loss: 0.001022
Epoch 9/30 Loss: 0.001003
Epoch 10/30 Loss: 0.000993
Epoch 11/30 Loss: 0.000987
Epoch 12/30 Loss: 0.000984
Epoch 13/30 Loss: 0.000973
Epoch 14/30 Loss: 0.000964
Epoch 15/30 Loss: 0.000968
Epoch 16/30 Loss: 0.000965
Epoch 17/30 Loss: 0.000963
Epoch 18/30 Loss: 0.000952
Epoch 19/30 Loss: 0.000965
Epoch 20/30 Loss: 0.000955
Epoch 21/30 Loss: 0.000950
Epoch 22/30 Loss: 0.000954
Epoch 23/30 Loss: 0.000947
Epoch 24/30 Loss: 0.000953
Epoch 25/30 Loss: 0.000954
Epoch 26/30 Loss: 0.000945
Epoch 27/30 Loss: 0.000945
Epoch 28/30 Loss: 0.000945
Epoch 29/30 Loss: 0.000946
Epoch 30/30 Loss: 0.000939


In [ ]:
train_time = time.time() - train_start
print("Time Taken:", train_time)

In [ ]:
torch.save(
    model.state_dict(),
    "CAH-STGCN-PEMS04.pth"
)

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np


test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

infer_start = time.time()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,    
    axis=0
)

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.01796092
RMSE: 0.03205648


In [ ]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)

print("MAPE:", mape)
print("R2:", r2)

In [ ]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)